# 2.1 Tool Calling Fundamentals

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langchain-google-genai==4.4.2" "langgraph==1.2.16"

## Your API key - pick ONE option

Some cells in this notebook call a model, and you need your own API key. Run only ONE of the next two cells, whichever fits you:

**Option A - Gemini (free, recommended if you have no OpenAI credit):** Google AI Studio gives a free API key, no card needed. Open https://aistudio.google.com/apikey , create a key, then run the Gemini cell below. The Flash models run on a free tier with rate limits.

**Option B - OpenAI (paid):** create a key at https://platform.openai.com/api-keys and run the OpenAI cell below.

Whichever key you paste is stored only in this notebook session, never saved to the file.

In [ ]:
# Option A: Gemini (free). Run this cell OR the OpenAI one, not both.
import os, getpass
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Paste your Gemini (AI Studio) API key (input stays hidden): ")

In [ ]:
# Option B: OpenAI (paid). Run this cell OR the Gemini one, not both.
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Using Gemini instead of OpenAI in the cells below

If you ran the Gemini key cell, one more small step: wherever a cell below creates the model with `ChatOpenAI(...)`, swap that creation line for:

```python
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
```

(Use whatever variable name the cell used, for example `model = ...` instead of `llm = ...`.) Everything else stays the same: prompts, chains, parsers and tools do not care which provider the model came from. For embeddings, swap `OpenAIEmbeddings` for `GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")` from the same package. On the OpenAI path, ignore this note and run the cells exactly as written.

## Learning objectives

- Describe the tool-calling round trip: the model emits a structured call, your code executes it, you feed the result back, the model answers.
- Attach tools to a chat model with `bind_tools` and explain what the model actually receives (JSON schemas, not Python functions).
- Read the `ToolCall` structure (`name`, `args`, `id`) off an `AIMessage` and execute calls with a tools-by-name lookup.
- Write the manual execution loop: the five-line pattern every agent framework runs under the hood.
- Choose between `tool_choice` values (`"auto"`, `"none"`, `"any"`, a specific tool name) and say when each is right.
- Explain the two most common misbehaviours, invented arguments and unneeded calls, and how to reduce them.
- State which models support tool calling well and why you verify per provider.

## The round trip

A language model cannot run your Python. It has no hands, only text. Tool calling is the protocol that bridges that gap, and it has exactly four steps:

1. **You describe your functions to the model** as JSON schemas: name, description, argument names and types. This is what `bind_tools` sends.
2. **The model replies with a structured tool call** instead of (or as well as) text: "call `get_weather` with `{"location": "Porto"}`".
3. **Your code runs the real function** with those arguments. The model never executes anything; you do.
4. **You send the result back** as a `ToolMessage`, and the model produces its final answer with the observation in context.

Nothing here is magic, and that is the point. When an agent framework "calls tools", it is running a loop of this shape. Lesson 2.3 (ReAct) and Module 4 (LangGraph) both build on this exact loop, so learn it bare before any framework hides it.

## bind_tools: attaching tools to a model

`bind_tools` takes tool-like objects (functions, Pydantic classes, `BaseTool` instances, or raw dicts) and returns a new runnable: the same model, with the tool schemas attached to every call it makes. Lesson 2.2 covers building tools properly with `@tool`; here is the smallest possible tool, just enough to make the mechanics visible:

In [ ]:
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"It is sunny in {location}."  # stub: a real tool would call a weather API

The decorator builds three things from the function: the name (`get_weather`), the description (the docstring), and the argument schema (from the type hints). The model will only ever see those three; the Python body is invisible to it.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
tools = [get_weather]
llm_with_tools = llm.bind_tools(tools)

The signature, verified against the pinned `langchain-openai` (shown for reference, not run):

```text
bind_tools(
    tools,                      # Sequence[dict | type | Callable | BaseTool]
    *,
    tool_choice=None,           # dict | str | bool | None
    strict=None,                # strict-mode structured outputs (OpenAI)
    parallel_tool_calls=None,   # allow several calls in one turn
    response_format=None,       # Pydantic schema for the final answer
    **kwargs,
)
```

`bind_tools` does not call anything. It returns a `Runnable` whose bound kwargs now include the OpenAI `tools` payload. You can inspect what the model will see with `llm_with_tools.kwargs["tools"]`: a list of `{"type": "function", "function": {...}}` dicts. When a tutorial's tool calls mysteriously fail, printing that payload is the fastest diagnostic: it shows exactly what the model was told.

## The ToolCall structure

Invoke the bound model on a question that needs the tool:

In [ ]:
from langchain_core.messages import HumanMessage

ai_msg = llm_with_tools.invoke([HumanMessage(content="What is the weather in Porto?")])
print(ai_msg.tool_calls)

A typical result:

In [ ]:
[{'name': 'get_weather',
  'args': {'location': 'Porto'},
  'id': 'call_abc123',
  'type': 'tool_call'}]

Each `ToolCall` is a dict with three required fields, verified against `langchain-core` 1.6.12:

- **`name`**: which tool the model wants. It must match a bound tool's name exactly.
- **`args`**: the arguments as a dict, already JSON-parsed by LangChain. You pass this dict straight to the tool.
- **`id`**: the provider's call identifier. You must echo it back in the `ToolMessage`, or the provider rejects the result as orphaned.

Note what the model did *not* produce: prose. `ai_msg.content` is usually an empty string on a tool-call turn. The decision lives entirely in `tool_calls`. Also note `ai_msg.invalid_tool_calls`: when the raw response contains a tool call whose arguments are not even valid JSON, LangChain cannot build a `ToolCall` and parks it there with the parse error attached, instead of crashing. Valid JSON with wrong *fields* is different: that lands in `tool_calls` and fails later, when the tool validates the args (more on that below).

## The manual execution loop

Here is the whole pattern. Five lines of loop body, and you now understand what every agent executor does:

In [ ]:
from langchain_core.messages import ToolMessage

tools_by_name = {t.name: t for t in tools}

messages = [HumanMessage(content="What is the weather in Porto?")]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)

for tc in ai_msg.tool_calls:                                  # 1. read each call
    result = tools_by_name[tc["name"]].invoke(tc["args"])     # 2. execute it
    messages.append(                                           # 3. feed the result back
        ToolMessage(content=str(result), tool_call_id=tc["id"])
    )

final = llm_with_tools.invoke(messages)                       # 4. model answers with the observation
print(final.content)
# It is sunny in Porto.

Four things to internalise:

1. **The lookup is yours.** `tools_by_name[tc["name"]]` is a plain dict. If the model names a tool you did not bind (it happens), this raises `KeyError`: handle it before this code meets users (lesson 2.4).
2. **The result must be a string.** `ToolMessage.content` is text. If your tool returns a dict, serialise it (`json.dumps` or `str()`); the model reads the observation as prose either way.
3. **`tool_call_id` is load-bearing.** It pairs each observation with its request. Omit it and the API returns an error about a missing tool call id.
4. **One loop is rarely enough.** Real agents repeat: invoke, execute, feed back, until a turn comes back with no `tool_calls`. Lesson 2.3 turns this into the ReAct loop with a step limit.

## Single vs parallel tool calls

A model can request several tools in one turn: `ai_msg.tool_calls` is a *list*. Ask "What is the weather in Porto and in Berlin?" with the weather tool bound, and you will typically get two calls in one message. The loop above already handles this: it iterates all calls, appends all observations, then re-invokes once.

Parallelism here means the model *planned* independent calls together, not that your code ran them concurrently. If the tools are slow and independent, execute them with threads or `asyncio`; if they depend on each other (the second needs the first's output), the model usually sequences them across turns on its own.

`parallel_tool_calls=False` in `bind_tools` tells the provider to allow at most one call per turn. Use it when your tools have side effects and you want the model to act step by step, or when a provider's parallel calls are unreliable. The default (`None`) leaves the provider's own default in place, which for OpenAI means parallel calls allowed.

## tool_choice: who decides

`tool_choice` controls how much freedom the model has. The options, with what LangChain actually sends to OpenAI (verified on the pinned versions):

| What you pass | What the model must do | Sent to the API |
|---|---|---|
| *(default)* `"auto"` | Decide freely, tool or plain text | `"auto"` |
| `"none"` | Never call a tool, answer in text | `"none"` |
| `"any"` (also `"required"` or `True`) | Call at least one tool, its pick | `"required"` |
| `"get_weather"` | Call exactly that tool | `{"type": "function", "function": {"name": "get_weather"}}` |
| dict `{"type": "function", "function": {"name": ...}}` | Same as naming it | passed through |

Defaults matter: leave `tool_choice` unset and the model answers from its own knowledge when it feels like it, which is correct for a general assistant. Force `"any"` when the turn is meaningless without a tool (a router that must classify, an agent mid-task). Force a *name* in tests and demos where you need a deterministic path, or when two tools overlap and the model keeps picking the wrong one. Force `"none"` when you want a pure text answer from a tool-bound model, for example a summary turn at the end of an agent run.

A warning that saves debugging time: `tool_choice="any"` guarantees *a* call, not a *sensible* call. A model forced to call something will call something, even inventing a use for a tool that does not fit. Forcing is a steering wheel, not a seatbelt.

## Why the model misbehaves

Two failure modes dominate every tool-calling project, and both are prompt problems wearing a technical disguise.

**Invented arguments.** The model calls a real tool with plausible-looking but wrong arguments: a date in the wrong format, a city id instead of a city name, a parameter the schema never defined. Causes, in order of frequency: the description never said what the argument should look like ("location" with no example), the schema is too loose (`str` where an enum or a format was needed), or the model is guessing because the user never supplied the value and no tool says "ask first". Fixes: write descriptions for the model, not for your code review (include formats and one example); prefer `Literal` enums and Pydantic field constraints over bare strings; and validate inside the tool so a bad call returns a useful error message the model can act on, instead of a stack trace.

**Unneeded or wrong-tool calls.** The model calls a tool for something it could have answered directly, or picks tool B when the job needed tool A. Causes: descriptions that overlap ("use this for weather" on two tools), missing "when NOT to use this" guidance, or `tool_choice="any"` forcing action where none was needed. Fix: make each description say when the tool applies *and* when it does not, keep the tool list short (a dozen vague tools choose worse than five sharp ones), and default to `"auto"`.

When a call is malformed beyond parsing, it lands in `invalid_tool_calls` with the error string: log those, because a rising rate means your schema or the model's behaviour drifted. When the JSON parses but the fields are wrong, `tool.invoke(tc["args"])` raises a validation error: catch it, and feed the error text back as the `ToolMessage` content so the model can correct itself on the next turn. Lesson 2.4 builds this retry into the loop.

## Which models support tool calling

Tool calling is a provider capability, not a LangChain one. OpenAI's chat models support it natively and well; that is why this course standardises on `gpt-4o-mini` for examples. Most major providers (Anthropic, Google, Mistral) expose their own flavour, and LangChain normalises them all into the same `ToolCall` structure, so the loop above does not change. But quality varies: smaller and cheaper models, and many local models, are noticeably worse at choosing tools and formatting arguments. Some providers emulate tool calling with JSON mode behind the scenes, which works until arguments get nested.

The practical rule: **verify per provider, per model.** Before building on a model, run the manual loop from this lesson against it with two or three known tool calls and check that names match, args validate, and ids round-trip. A model that scores well on chat benchmarks can still be sloppy at function calling. Keep `temperature=0` for tool-calling turns: determinism helps both testing and production behaviour.

## Where this goes next

- **2.2** builds the tools themselves properly: the `@tool` decorator in depth, explicit Pydantic arg schemas, and writing descriptions the model actually follows.
- **2.3** wraps this chapter's loop in the ReAct pattern: Thought, Action, Observation, repeated with a step limit.
- **2.4** hardens the loop: unknown tool names, validation errors fed back for self-correction, retries, and max-step guards.
- **Module 4** rebuilds the same loop as a LangGraph graph, where the "execute tools" step becomes a graph node.

## What can go wrong

- **The model never calls the tool.** With `tool_choice` at its default `"auto"`, the model may answer from knowledge instead. If the turn requires a tool, say so in the user message or set `tool_choice="any"`. Also check the description: a vague description is a tool the model cannot see a use for.
- **KeyError on the tool name.** The model invents or misspells a tool name and `tools_by_name[tc["name"]]` blows up. Never let a raw model output index your dict unguarded; fall back to an error observation (lesson 2.4 wires this in).
- **Orphaned ToolMessage.** Forgetting `tool_call_id`, or pairing it with the wrong call when several ran in parallel, makes the provider reject the follow-up call. Build the `ToolMessage` right next to the execution, in the same loop iteration.
- **Non-string observations.** Passing a dict or None as `ToolMessage` content raises or serialises badly. Normalise with `str()` or `json.dumps()` at the boundary, every time.
- **Forced calls that make no sense.** `tool_choice="any"` on a turn with no good tool produces confident nonsense with extra steps. Force only when the turn genuinely needs a tool.
- **Silent provider differences.** Code tested on OpenAI can behave differently on another provider: parallel calls merged, ids formatted differently, or tool calling emulated through JSON mode. Re-run the manual loop when you change providers; do not assume parity.
- **Cost of the round trip.** Each loop iteration is another full model call, with the growing message list (including every tool observation) re-sent each time. Verbose tool outputs inflate input tokens fast; keep observations compact.

## Check your understanding

**1. In the four-step round trip, which steps does the model perform and which does your code perform?**
The model performs steps 1 (receiving the schemas) and 2 (emitting the structured call) and the final answer; your code performs step 3 (executing the real function) and step 4 (packaging the result as a `ToolMessage`). The model never runs your code.

**2. What are the three required fields of a `ToolCall`, and which one must you echo back?**
`name` (which tool), `args` (the arguments dict), and `id` (the provider's call identifier). The `id` must be echoed back as `tool_call_id` in the `ToolMessage`, or the provider rejects the result as orphaned.

**3. What is the difference between a call landing in `invalid_tool_calls` and one that fails at `tool.invoke(tc["args"])`?**
`invalid_tool_calls` holds calls whose raw arguments were not even valid JSON, so LangChain could not build a `ToolCall`. A call that parses but has wrong fields lands in `tool_calls` and fails later at `tool.invoke`, when the tool's schema validation rejects the args.

**4. You bind three tools and pass `tool_choice="get_weather"`. The user asks for a joke. What happens, and what does this teach about forcing?**
The model must call `get_weather`; it will invent a location and call it anyway. Forcing guarantees a call, not a sensible one: use a specific tool name only when the turn genuinely needs that tool.

**5. Your agent works on OpenAI but picks wrong tools on a cheaper provider. Name two things to check before blaming your code.**
First, re-run this lesson's manual loop against the new provider with known tool calls, checking name matches, arg validation, and id round-trips. Second, check whether the provider emulates tool calling through JSON mode and whether `parallel_tool_calls` behaves the same; provider parity cannot be assumed.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.